# Decision Tree

Binary classification on California housing: `median_house_value > median`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_df = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

threshold = data_df['median_house_value'].median()
y = (data_df['median_house_value'] > threshold).astype(int).to_numpy()
X = data_df.drop(columns=['median_house_value']).to_numpy()
feature_names = list(data_df.drop(columns=['median_house_value']).columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, shuffle=True, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(np.bincount(y_train), np.bincount(y_test))

## Baseline

In [ ]:
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Train acc: {accuracy_score(y_train, clf.predict(X_train)):.4f}")
print(f"Test acc:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Tree depth: {clf.get_depth()}")
print(f"Leaves: {clf.get_n_leaves()}")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Below', 'Above']))

## Depth sweep

In [ ]:
depths = list(range(1, 26))
train_acc, test_acc = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, m.predict(X_train)))
    test_acc.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.plot(depths, train_acc, 'o-', label='train')
plt.plot(depths, test_acc, 's-', label='test')
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_d = depths[int(np.argmax(test_acc))]
print(f"Best depth: {best_d}, test acc: {max(test_acc):.4f}")

## Splitting criterion

In [ ]:
for crit in ['gini', 'entropy', 'log_loss']:
    m = DecisionTreeClassifier(criterion=crit, max_depth=best_d, random_state=42)
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    print(f"{crit:<10} train={tr:.4f} test={te:.4f}")

## Feature importance

In [ ]:
best_tree = DecisionTreeClassifier(max_depth=best_d, random_state=42).fit(X_train, y_train)
importance = pd.Series(best_tree.feature_importances_, index=feature_names).sort_values()

plt.figure(figsize=(9, 6))
importance.plot(kind='barh', color='steelblue')
plt.xlabel('importance')
plt.grid(alpha=0.3)
plt.show()

print(importance)

## Tree visualization (shallow for readability)

In [ ]:
small = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
plt.figure(figsize=(20, 10))
plot_tree(small, feature_names=feature_names, class_names=['Below', 'Above'],
          filled=True, rounded=True, fontsize=10)
plt.show()